# Pipeline FINAL UNIVERSAL (STRICT)

Ce notebook est conçu pour **marcher sur n’importe quel fichier** (assurance / e-commerce / SaaS / autre) en gardant une **taxonomie universelle** + un **lexique sectoriel optionnel**.

Il génère des sorties **Streamlit-safe** :

- `data/commentaires_topics.csv` : 1 ligne par texte, avec `theme`
- `data/resultats_analyse_commentaires.csv` : agrégé `theme,count`

🔒 Règle : **`theme` = catégorie métier** (jamais des trigrams).

---

## Fonctionnement
- Détecte (ou impose) la colonne texte
- Détecte (ou impose) le secteur (`SECTOR`)
- `incident` + `sentiment_pred`
- Catégorisation macro **hybride** (keywords + embeddings si dispo)
- **Inconnu** si faible confiance et aucun signal
- **Autre** si non-incident
- Multi-label (secondaire si proche)
- Audit (low_confidence / Inconnu / contradictions)


## 1) Imports

In [1]:

PIPELINE_VERSION = "UNIVERSAL_STRICT_V1"

# ============
# Entrées / sorties
# ============
INPUT_PATH = "data/commentaires_assurance_auto.csv"  # <-- change si besoin
OUTPUT_TOPICS_PATH = "data/commentaires_topics.csv"
OUTPUT_AGG_PATH    = "data/resultats_analyse_commentaires.csv"

# ============
# Colonne texte
# ============
TEXT_COL = None  # None = auto-détection (recommandé). Sinon ex: "commentaire" / "verbatim" / "text"

# ============
# Secteur (manuel ou auto)
# ============
SECTOR = "auto"  # "auto" | "assurance" | "ecommerce" | "saas" | "generic"

# ============
# Seuils
# ============
EMB_THRESHOLD = 0.35
SECONDARY_RATIO = 0.95
SUPERVISED_THRESHOLD = 0.55  # réservé si tu ajoutes un labels.csv plus tard

# ============
# Format CSV
# ============
CSV_SEP = None   # None = auto (python engine). Mets ";" si CSV Excel FR
HAS_HEADER = "auto"  # "auto" | True | False


In [4]:

import os
import re
import time
import numpy as np
import pandas as pd

print("PIPELINE:", PIPELINE_VERSION)
print("INPUT:", INPUT_PATH)


PIPELINE: UNIVERSAL_STRICT_V1
INPUT: data/commentaires_assurance_auto.csv


## 2) Lecture CSV robuste (auto)

- Si ton fichier est un CSV Excel FR, mets `CSV_SEP=';'`.
- Si ton fichier n’a pas de header, mets `HAS_HEADER=False`.

⚠️ Pour ton fichier assurance historique (sans header), tu peux forcer :
- `CSV_SEP=';'`
- `HAS_HEADER=False`
- et éventuellement `COLUMNS_ASSURANCE` (voir cellule suivante).

In [5]:

def read_csv_robust(path: str, sep=None, has_header="auto", names=None):
    # header
    if has_header == "auto":
        header = "infer"
    elif has_header is True:
        header = "infer"
    else:
        header = None

    df = pd.read_csv(
        path,
        sep=sep,
        header=header,
        names=names,
        engine="python",
        encoding="utf-8",
        encoding_errors="replace",
        on_bad_lines="skip",
    )
    return df

df = read_csv_robust(INPUT_PATH, sep=CSV_SEP, has_header=HAS_HEADER)

print("Lignes:", len(df))
print("Colonnes:", list(df.columns))
df.head(3)


Lignes: 8310
Colonnes: ['1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent', ' les photos restent en attente à la fin du formulaire malgré plusieurs essais', " du coup rien n'avance", ' merci quand même.']


,1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent,les photos restent en attente à la fin du formulaire malgré plusieurs essais,du coup rien n'avance,merci quand même.
0,2;2024-03-22;iOS;App;neg;complexité;collision;...,c'est trop technique pour un sinistre simple ...,franchement je perds un temps fou,c'est pas sérieux.
1,3;2024-04-26;iOS;Email;pos;authentification;co...,ça bloque sur l'authentification quand je val...,mais tout est bien expliqué,bravo à l'équipe.
2,4;2024-04-22;iOS;Web;neu;photo;collision;1-3 a...,l'envoi des clichés échoue sur l'étape des ju...,honnêtement j'ai dû m'y reprendre,j'attends une amélioration.


## 2bis) Option : mapping fixe pour TON ancien CSV assurance (sans header)

Si ton fichier est exactement le format historique (14 colonnes, `sep=';'`, sans header), active cette cellule en décommentant.

In [6]:

# === Décommenter si besoin pour ton ancien format assurance ===
# COLUMNS_ASSURANCE = [
#     "id", "date_commentaire", "device", "canal", "sentiment_source",
#     "type_irritant", "type_sinistre", "anciennete_client",
#     "type_contrat", "csat", "gravite_sinistre",
#     "region_client", "canal_resolution", "commentaire"
# ]
# df = read_csv_robust(INPUT_PATH, sep=";", has_header=False, names=COLUMNS_ASSURANCE)
# print("Lignes:", len(df))
# print("Colonnes:", list(df.columns))
# df.head(3)


## 3) Auto-détection de la colonne texte

On cherche une colonne qui ressemble à : `commentaire`, `verbatim`, `texte`, `review`, `message`, etc.

Tu peux forcer via `TEXT_COL = 'commentaire'` dans les paramètres.

In [7]:

TEXT_CANDIDATES = [
    "commentaire", "verbatim", "texte", "text", "review", "message", "content",
    "avis", "feedback", "description"
]

def detect_text_col(df: pd.DataFrame):
    cols = [c for c in df.columns]
    cols_lower = {str(c).lower(): c for c in cols}
    for cand in TEXT_CANDIDATES:
        if cand in cols_lower:
            return cols_lower[cand]
    # fallback: colonne object la plus longue en moyenne
    obj_cols = [c for c in cols if df[c].dtype == "object"]
    if not obj_cols:
        raise ValueError("Aucune colonne texte détectée. Force TEXT_COL.")
    lengths = {c: df[c].astype(str).str.len().mean() for c in obj_cols}
    return max(lengths.items(), key=lambda x: x[1])[0]

if TEXT_COL is None:
    TEXT_COL = detect_text_col(df)

print("✅ Colonne texte utilisée:", TEXT_COL)
df[[TEXT_COL]].head(3)


✅ Colonne texte utilisée: 1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent


,1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent
0,2;2024-03-22;iOS;App;neg;complexité;collision;...
1,3;2024-04-26;iOS;Email;pos;authentification;co...
2,4;2024-04-22;iOS;Web;neu;photo;collision;1-3 a...


## 4) Normalisation texte (stable)

In [8]:

def normalize_text(t: str) -> str:
    if pd.isna(t):
        return ""
    t = str(t).lower().replace("’", "'")
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[\d]+", " ", t)
    t = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ'\s-]", " ", t)
    t = re.sub(r"[-_]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text_norm"] = df[TEXT_COL].astype(str).fillna("").apply(normalize_text)
df[[TEXT_COL, "text_norm"]].head(5)


,1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent,text_norm
0,2;2024-03-22;iOS;App;neg;complexité;collision;...,ios app neg complexité collision an tiers moye...
1,3;2024-04-26;iOS;Email;pos;authentification;co...,ios email pos authentification collision ans j...
2,4;2024-04-22;iOS;Web;neu;photo;collision;1-3 a...,ios web neu photo collision ans tiers faible b...
3,5;2024-10-02;Android;Web;pos;lenteur;collision...,android web pos lenteur collision ans tous ris...
4,6;2024-01-06;iOS;App;pos;bug;vol;1-3 ans;tiers...,ios app pos bug vol ans tiers faible bourgogne...


## 5) Sentiment + incident (robuste, cross-secteurs)

- `incident=True` si plainte explicite (impossible, bug, erreur, problème, etc.)
- `sentiment_pred` simple: Positif / Négatif / Neutre

Garde-fou : si clairement positif, `incident=False`.

In [9]:

POSITIVE_KW = [
    r"\bparfait\b", r"\bexcellent\b", r"\bgénial\b", r"\bgenial\b", r"\btop\b",
    r"\bmerci\b", r"\brecommande\b", r"\btrès bien\b", r"\btres bien\b",
    r"\bsatisfait\b", r"\befficace\b", r"\brapide\b", r"\bfluide\b"
]
NEGATIVE_KW = [
    r"\bprobl[eè]me\b", r"\bbug\b", r"\berreur\b", r"\bimpossible\b",
    r"\bne fonctionne pas\b", r"\bne marche pas\b", r"\b(réponse|reponse) (trop )?long\b",
    r"\blent(e|eur)?\b", r"\bbloqu(e|é|ée)\b", r"\bplant(e|é)\b",
    r"\bdéçu\b", r"\bdecu\b", r"\bnul\b", r"\bmécontent\b", r"\bmecontent\b",
    r"\baucune r(e|é)ponse\b", r"\bpas de r(e|é)ponse\b"
]

def detect_sentiment_and_incident(text: str):
    t = normalize_text(text)
    pos = sum(1 for p in POSITIVE_KW if re.search(p, t))
    neg = sum(1 for p in NEGATIVE_KW if re.search(p, t))
    incident = (neg >= 1)

    if neg >= 1 and neg >= pos:
        return "Négatif", float(neg), True
    if pos >= 1 and pos > neg:
        return "Positif", float(pos), False
    return "Neutre", 0.0, incident

tmp = df[TEXT_COL].apply(lambda x: pd.Series(detect_sentiment_and_incident(x)))
df["sentiment_pred"], df["score_sentiment"], df["incident"] = tmp[0], tmp[1], tmp[2]

df[[TEXT_COL, "sentiment_pred", "incident"]].head(10)


,1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent,sentiment_pred,incident
0,2;2024-03-22;iOS;App;neg;complexité;collision;...,Négatif,True
1,3;2024-04-26;iOS;Email;pos;authentification;co...,Positif,False
2,4;2024-04-22;iOS;Web;neu;photo;collision;1-3 a...,Neutre,False
3,5;2024-10-02;Android;Web;pos;lenteur;collision...,Négatif,True
4,6;2024-01-06;iOS;App;pos;bug;vol;1-3 ans;tiers...,Négatif,True
5,7;2024-09-28;Web;Web;neg;photo;collision;10+ a...,Négatif,True
6,8;2024-09-27;Web;App;neg;parcours;bris_de_glac...,Négatif,True
7,10;2024-12-10;iOS;Web;neg;bug;vol;0-1 an;tiers...,Négatif,True
8,11;2024-03-26;Android;App;pos;authentification...,Positif,False
9,12;2024-01-30;Android;App;pos;parcours;collisi...,Positif,False


## 6) Taxonomie universelle + lexiques sectoriels (boost optionnel)

### Universelle (toujours active)
- Technique/Bug
- Connexion/Compte
- Paiement/Facturation/Abonnement
- Support/Service client
- Performance/Lenteur
- Données/Synchronisation

### Sectoriel (optionnel)
- Assurance : Sinistre, Contrat/Attestation
- E-commerce : Livraison, Commande/Produit
- SaaS : API/Intégration, Permissions/Accès, Onboarding


In [10]:

UNIVERSAL = [
    {"name":"Technique / Bug / Fonctionnalité",
     "desc":"bug, erreur, crash, application, site, fonctionnalité ne marche pas, écran bloqué, incident technique",
     "kw":[r"\bbug\b", r"\berreur\b", r"\bcrash\b", r"\bplante\b", r"\bplant(e|é)\b",
           r"\bimpossible\b", r"\bne fonctionne pas\b", r"\bne marche pas\b", r"\bbloqu(e|é|ée|és)\b",
           r"\bapplication\b", r"\bsite\b", r"\bfonctionnalit\w+\b"]},

    {"name":"Compte / Connexion / Authentification",
     "desc":"connexion, authentification, login, mot de passe, code sms, accès, compte bloqué, espace client",
     "kw":[r"\bconnexion\b", r"\bconnect(er|ion)\b", r"\bauthentif\w+\b", r"\blogin\b",
           r"\bmot de passe\b", r"\bcode\b", r"\bsms\b", r"\bac(c|ç)ès\b", r"\bcompte\b",
           r"\bespace client\b"]},

    {"name":"Paiement / Facturation / Abonnement",
     "desc":"paiement, carte refusée, facturation, abonnement, prélèvement, débit, remboursement, renouvellement",
     "kw":[r"\bpaiement\b", r"\bcarte\b", r"\brefus(é|ée|ee)\b", r"\bfactur\w+\b",
           r"\babonn\w+\b", r"\bprélèv\w+\b", r"\bprelev\w+\b", r"\bdébit\b", r"\bdebit\b",
           r"\brembours\w+\b", r"\brenouvel\w+\b"]},

    {"name":"Support / Service client",
     "desc":"support, service client, assistance, ticket, conseiller, sav, contact, aucune réponse",
     "kw":[r"\bsupport\b", r"\bservice client\b", r"\bassistance\b", r"\bticket\b",
           r"\bconseiller\b", r"\bsav\b", r"\bcontact\b", r"\baucune r(e|é)ponse\b",
           r"\bpas de r(e|é)ponse\b"]},

    {"name":"Performance / Lenteur",
     "desc":"lent, lenteur, latence, chargement, attente, rame, performance dégradée",
     "kw":[r"\blent(e|eur|ement)?\b", r"\blenteur\b", r"\blatence\b",
           r"\bchargement\b", r"\battente\b", r"\brame\b", r"\bperformance\b"]},

    {"name":"Données / Synchronisation",
     "desc":"données perdues, informations disparues, historique supprimé, synchronisation, sauvegarde",
     "kw":[r"\bdonn(é|ee)es?\b", r"\binformation(s)?\b", r"\bdisparu(e|es)?\b",
           r"\bperdu(e|es)?\b", r"\bhistorique\b", r"\bsynchron\w+\b", r"\bsauvegard\w+\b"]},
]

ASSURANCE = [
    {"name":"Sinistre / Indemnisation",
     "desc":"sinistre, indemnisation, expertise, réparation, prise en charge, dossier, remboursement sinistre",
     "kw":[r"\bsinistre\b", r"\bindemn\w+\b", r"\bexpertise\b", r"\bréparation\b", r"\breparation\b",
           r"\bprise en charge\b", r"\bdossier\b"]},

    {"name":"Contrat / Attestation / Documents",
     "desc":"contrat, attestation, documents, justificatif, avenant, résiliation, carte verte",
     "kw":[r"\bcontrat\b", r"\battestation\b", r"\bdocument(s)?\b", r"\bjustificatif\b",
           r"\bavenant\b", r"\brésili\w+\b", r"\bresili\w+\b", r"\bcarte verte\b"]},
]

ECOMMERCE = [
    {"name":"Livraison / Logistique",
     "desc":"livraison, colis, suivi, retard, transporteur, point relais, non reçu, livré mais non reçu",
     "kw":[r"\blivraison\b", r"\bcolis\b", r"\bsuivi\b", r"\bretard\b",
           r"\btransport\w+\b", r"\bpoint relais\b", r"\bnon re(ç|c)u\b",
           r"\bmarqu(é|ee) livr(é|ee)\b"]},

    {"name":"Commande / Panier / Produit",
     "desc":"commande, panier, validation, produit, article, stock, retour produit",
     "kw":[r"\bcommande\b", r"\bpanier\b", r"\bvalider\b", r"\bvalidation\b", r"\bproduit\b",
           r"\barticle\b", r"\bstock\b", r"\bretour\b"]},
]

SAAS = [
    {"name":"API / Intégration",
     "desc":"api, intégration, webhook, connecteur, sso, oauth, documentation technique",
     "kw":[r"\bapi\b", r"\bintégration\b", r"\bintegration\b", r"\bwebhook\b",
           r"\bconnecteur\b", r"\bsso\b", r"\boauth\b", r"\bdocumentation\b"]},

    {"name":"Permissions / Accès / Rôles",
     "desc":"permissions, rôles, droits, accès équipe, utilisateurs, admin, autorisation",
     "kw":[r"\bpermission(s)?\b", r"\brôle(s)?\b", r"\brole(s)?\b", r"\bdroit(s)?\b",
           r"\butilisateur(s)?\b", r"\badmin\b", r"\bautorisation\b"]},

    {"name":"Onboarding / Paramétrage",
     "desc":"onboarding, configuration, paramétrage, mise en place, installation, setup",
     "kw":[r"\bonboarding\b", r"\bconfigur\w+\b", r"\bparamétr\w+\b", r"\bparametr\w+\b",
           r"\binstallation\b", r"\bsetup\b"]},
]

def detect_sector(df: pd.DataFrame):
    # Heuristique sur texte_norm (échantillon)
    sample = df["text_norm"].dropna().astype(str).head(500)
    text = " ".join(sample.tolist())
    scores = {"assurance":0, "ecommerce":0, "saas":0}
    for w in ["sinistre","attestation","contrat","indemnisation","franchise"]:
        scores["assurance"] += text.count(w)
    for w in ["livraison","colis","transporteur","commande","panier","retour"]:
        scores["ecommerce"] += text.count(w)
    for w in ["api","intégration","integration","sso","webhook","oauth","onboarding","admin","permissions"]:
        scores["saas"] += text.count(w)
    best = max(scores.items(), key=lambda x: x[1])
    if best[1] == 0:
        return "generic"
    return best[0]

if SECTOR == "auto":
    SECTOR = detect_sector(df)

print("✅ Secteur retenu:", SECTOR)

CATEGORIES = list(UNIVERSAL)  # base universelle toujours
if SECTOR == "assurance":
    CATEGORIES += ASSURANCE
elif SECTOR == "ecommerce":
    CATEGORIES += ECOMMERCE
elif SECTOR == "saas":
    CATEGORIES += SAAS
else:
    pass  # generic

print("Nombre de catégories:", len(CATEGORIES))
[c["name"] for c in CATEGORIES]


✅ Secteur retenu: assurance
Nombre de catégories: 8


['Technique / Bug / Fonctionnalité',
 'Compte / Connexion / Authentification',
 'Paiement / Facturation / Abonnement',
 'Support / Service client',
 'Performance / Lenteur',
 'Données / Synchronisation',
 'Sinistre / Indemnisation',
 'Contrat / Attestation / Documents']

## 7) Vote keywords (avec boost sectoriel implicite)

On garde simple : le boost sectoriel est porté par les catégories ajoutées (assurance/ecommerce/saas).

In [11]:

def keyword_vote(text: str):
    t = normalize_text(text)
    scores = {}
    for cat in CATEGORIES:
        hit = sum(1 for p in cat["kw"] if re.search(p, t))
        if hit:
            scores[cat["name"]] = hit
    if not scores:
        return None, 0
    best = max(scores.items(), key=lambda x: x[1])
    return best[0], best[1]


## 8) Embeddings (optionnel) + multi-label + garde-fous

- Si embeddings dispo : similarité vers descriptions des catégories.
- Sinon : keywords only.
- `Inconnu` si faible confiance ET pas de keywords.
- `Autre` si non-incident.


In [12]:

def build_embedder():
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    except Exception as e:
        print("⚠️ Embeddings indisponibles -> mode keywords only.")
        return None

def embed_match(text: str, embedder, cat_vecs):
    v = embedder.encode([text], normalize_embeddings=True)[0]
    sims = np.dot(cat_vecs, v)
    idx = np.argsort(-sims)
    b, s = int(idx[0]), float(sims[idx[0]])
    b2, s2 = int(idx[1]), float(sims[idx[1]]) if len(idx) > 1 else (b, s)
    return b, s, b2, s2

embedder = build_embedder()
cat_vecs = None
if embedder is not None:
    cat_descs = [c["desc"] for c in CATEGORIES]
    cat_vecs = embedder.encode(cat_descs, normalize_embeddings=True)

cats_main, cats_second, scores, methods = [], [], [], []

for txt, is_inc in zip(df[TEXT_COL].tolist(), df["incident"].tolist()):
    kw_cat, kw_score = keyword_vote(txt)

    if not is_inc:
        cats_main.append("Autre")
        cats_second.append("")
        scores.append(0.0)
        methods.append("non_incident")
        continue

    if embedder is not None and cat_vecs is not None:
        b, s, b2, s2 = embed_match(normalize_text(txt), embedder, cat_vecs)
        main = CATEGORIES[b]["name"]
        second = CATEGORIES[b2]["name"] if (s2 >= SECONDARY_RATIO*s and CATEGORIES[b2]["name"] != main) else ""

        if s < EMB_THRESHOLD:
            if kw_cat is not None:
                cats_main.append(kw_cat); cats_second.append("")
                scores.append(float(kw_score)); methods.append("keywords_fallback")
            else:
                cats_main.append("Inconnu"); cats_second.append("")
                scores.append(s); methods.append("low_confidence")
        else:
            cats_main.append(main); cats_second.append(second)
            scores.append(s); methods.append("embeddings")
    else:
        if kw_cat is not None:
            cats_main.append(kw_cat); cats_second.append("")
            scores.append(float(kw_score)); methods.append("keywords")
        else:
            cats_main.append("Inconnu"); cats_second.append("")
            scores.append(0.0); methods.append("no_signal")

df["categorie_principale"] = cats_main
df["categorie_secondaire"] = cats_second
df["score_categorie"] = scores
df["methode_categorie"] = methods

df[[TEXT_COL,"incident","categorie_principale","categorie_secondaire","score_categorie","methode_categorie"]].head(15)


c:\Users\Malaka\Desktop\Commentairecsv_v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent,incident,categorie_principale,categorie_secondaire,score_categorie,methode_categorie
0,2;2024-03-22;iOS;App;neg;complexité;collision;...,True,Inconnu,,0.267939,low_confidence
1,3;2024-04-26;iOS;Email;pos;authentification;co...,False,Autre,,0.000000,non_incident
2,4;2024-04-22;iOS;Web;neu;photo;collision;1-3 a...,False,Autre,,0.000000,non_incident
3,5;2024-10-02;Android;Web;pos;lenteur;collision...,True,Performance / Lenteur,Technique / Bug / Fonctionnalité,0.364402,embeddings
4,6;2024-01-06;iOS;App;pos;bug;vol;1-3 ans;tiers...,True,Technique / Bug / Fonctionnalité,,0.466760,embeddings
5,7;2024-09-28;Web;Web;neg;photo;collision;10+ a...,True,Inconnu,,0.213229,low_confidence
6,8;2024-09-27;Web;App;neg;parcours;bris_de_glac...,True,Inconnu,,0.295794,low_confidence
7,10;2024-12-10;iOS;Web;neg;bug;vol;0-1 an;tiers...,True,Technique / Bug / Fonctionnalité,,0.482712,embeddings
8,11;2024-03-26;Android;App;pos;authentification...,False,Autre,,0.000000,non_incident
9,12;2024-01-30;Android;App;pos;parcours;collisi...,False,Autre,,0.000000,non_incident


## 9) Audit qualité (top 30)

- low_confidence
- Inconnu
- contradictions (Positif mais classé en problème)


In [13]:

df["contradiction"] = (
    (df["sentiment_pred"] == "Positif") &
    (df["categorie_principale"].isin([c["name"] for c in CATEGORIES]))
)

audit_low = df[df["methode_categorie"].eq("low_confidence")].head(30)
audit_inc = df[df["categorie_principale"].eq("Inconnu")].head(30)
audit_contra = df[df["contradiction"]].head(30)

print("low_confidence:", len(audit_low), "| Inconnu:", len(audit_inc), "| contradictions:", len(audit_contra))

audit_low[[TEXT_COL,"sentiment_pred","incident","categorie_principale","score_categorie","methode_categorie"]]


low_confidence: 30 | Inconnu: 30 | contradictions: 0


,1;2024-11-23;Android;Web;neg;photo;collision;3-10 ans;tous_risques;2;faible;Île-de-France;App;Très mécontent,sentiment_pred,incident,categorie_principale,score_categorie,methode_categorie
0,2;2024-03-22;iOS;App;neg;complexité;collision;...,Négatif,True,Inconnu,0.267939,low_confidence
5,7;2024-09-28;Web;Web;neg;photo;collision;10+ a...,Négatif,True,Inconnu,0.213229,low_confidence
6,8;2024-09-27;Web;App;neg;parcours;bris_de_glac...,Négatif,True,Inconnu,0.295794,low_confidence
11,14;2024-03-21;Android;App;neg;complexité;vol;3...,Négatif,True,Inconnu,0.314286,low_confidence
13,16;2024-03-17;iOS;Web;neg;photo;collision;1-3 ...,Négatif,True,Inconnu,0.213280,low_confidence
15,19;2024-10-22;Android;App;neg;parcours;vol;3-1...,Négatif,True,Inconnu,0.327783,low_confidence
34,39;2024-01-01;iOS;App;neg;photo;collision;3-10...,Négatif,True,Inconnu,0.273425,low_confidence
49,54;2024-09-19;iOS;App;neg;parcours;bris_de_gla...,Négatif,True,Inconnu,0.286962,low_confidence
51,56;2024-02-18;Android;App;neg;complexité;bris_...,Négatif,True,Inconnu,0.245257,low_confidence
54,59;2024-06-15;iOS;Web;neg;photo;collision;3-10...,Négatif,True,Inconnu,0.284802,low_confidence


## 10) Exports Streamlit-safe

🔥 On force : `theme = categorie_principale`.

- `commentaires_topics.csv` : détails par texte
- `resultats_analyse_commentaires.csv` : agrégé


In [17]:
from pathlib import Path
import pandas as pd

# ============================================================
# EXPORT STREAMLIT-SAFE (standard industrialisable)
# ============================================================
def export_streamlit_safe(df: pd.DataFrame, out_path: str | Path) -> None:
    """
    Export CSV stable pour Streamlit / Cloud :
    - sep=',' (standard)
    - encoding='utf-8'
    - header OK
    - index=False
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    df_out = df.copy()
    df_out.columns = [str(c).strip() for c in df_out.columns]  # noms clean

    df_out.to_csv(out_path, index=False, sep=",", encoding="utf-8")


# ============================================================
# 1) GARANTIR LA COLONNE TEXTE CANONIQUE : 'texte'
# ============================================================
TEXT_CANDIDATES = ["commentaire", "verbatim", "text", "message", "contenu", "text_norm"]

if "texte" not in df.columns:
    source_col = next((c for c in TEXT_CANDIDATES if c in df.columns), None)
    if source_col is None:
        raise ValueError(
            "Impossible de créer la colonne 'texte'. "
            f"Aucune colonne trouvée parmi: {TEXT_CANDIDATES}. "
            f"Colonnes actuelles: {list(df.columns)}"
        )
    df["texte"] = df[source_col].astype(str)


# ============================================================
# 2) RÈGLE ABSOLUE : theme = categorie_principale
# ============================================================
if "categorie_principale" not in df.columns:
    raise ValueError(
        "categorie_principale manquante : impossible de définir theme. "
        f"Colonnes actuelles: {list(df.columns)}"
    )

df["theme"] = df["categorie_principale"].astype(str)


# ============================================================
# 3) GARDE-FOUS (cohérence)
# ============================================================
# Autre => non-incident
if "incident" in df.columns:
    df.loc[df["theme"].str.lower().eq("autre"), "incident"] = False

# Inconnu => low_confidence True (si dispo)
if "low_confidence" in df.columns:
    df.loc[df["theme"].str.lower().isin(["inconnu", "unknown"]), "low_confidence"] = True


# ============================================================
# 4) EXPORT 1 : commentaires_topics.csv (viewer Streamlit)
# ============================================================
TOPICS_COLS = [
    "texte",
    "text_norm",
    "sentiment", "sentiment_pred", "score_sentiment",
    "incident",
    "categorie_principale", "categorie_secondaire",
    "theme",
    "confidence", "score_categorie",
    "methode_categorie",
    "low_confidence",
    "contradiction",
]

topics_cols = [c for c in TOPICS_COLS if c in df.columns]
df_topics = df[topics_cols].copy()

export_streamlit_safe(df_topics, "data/commentaires_topics.csv")


# ============================================================
# 5) EXPORT 2 : resultats_analyse_commentaires.csv (agrégats)
# ============================================================
# Stats par thème (volume, incidents, etc.)
if "incident" in df.columns:
    df_results = (
        df.groupby("theme", dropna=False)
          .agg(
              nb=("theme", "size"),
              nb_incidents=("incident", "sum"),
              part_incidents=("incident", "mean"),
          )
          .reset_index()
          .sort_values("nb", ascending=False)
    )
else:
    df_results = (
        df.groupby("theme", dropna=False)
          .agg(nb=("theme", "size"))
          .reset_index()
          .sort_values("nb", ascending=False)
    )

export_streamlit_safe(df_results, "data/resultats_analyse_commentaires.csv")


# ============================================================
# 6) TEST DE LECTURE (comme Streamlit)
# ============================================================
t = pd.read_csv("data/commentaires_topics.csv")
r = pd.read_csv("data/resultats_analyse_commentaires.csv")

print("✅ commentaires_topics.csv -> nb colonnes:", len(t.columns), "| colonnes:", t.columns.tolist())
print("✅ resultats_analyse_commentaires.csv -> nb colonnes:", len(r.columns), "| colonnes:", r.columns.tolist())

# Sanity : doit afficher texte + theme
assert "texte" in t.columns, "❌ 'texte' manquant dans commentaires_topics.csv"
assert "theme" in t.columns, "❌ 'theme' manquant dans commentaires_topics.csv"
print("✅ OK : exports Streamlit-safe")


✅ commentaires_topics.csv -> nb colonnes: 11 | colonnes: ['texte', 'text_norm', 'sentiment_pred', 'score_sentiment', 'incident', 'categorie_principale', 'categorie_secondaire', 'theme', 'score_categorie', 'methode_categorie', 'contradiction']
✅ resultats_analyse_commentaires.csv -> nb colonnes: 4 | colonnes: ['theme', 'nb', 'nb_incidents', 'part_incidents']
✅ OK : exports Streamlit-safe


## 11) (Option) Générer un template `labels_template.csv`

Pour feedback humain léger / apprentissage supervisé ultérieur.

In [19]:

LABELS_TEMPLATE_PATH = "data/labels_template.csv"

cand = df_export[df_export["theme"].isin(["Inconnu"])].copy().head(300)
labels_template = cand[[TEXT_COL]].copy()
labels_template.columns = ["commentaire"]  # standardise
labels_template["categorie_true"] = ""

labels_template.to_csv(LABELS_TEMPLATE_PATH, sep=";", index=False)
print("✅ Export:", LABELS_TEMPLATE_PATH, "| lignes:", len(labels_template))

labels_template.head(10)


✅ Export: data/labels_template.csv | lignes: 300


,commentaire,categorie_true
0,2;2024-03-22;iOS;App;neg;complexité;collision;...,
5,7;2024-09-28;Web;Web;neg;photo;collision;10+ a...,
6,8;2024-09-27;Web;App;neg;parcours;bris_de_glac...,
11,14;2024-03-21;Android;App;neg;complexité;vol;3...,
13,16;2024-03-17;iOS;Web;neg;photo;collision;1-3 ...,
15,19;2024-10-22;Android;App;neg;parcours;vol;3-1...,
34,39;2024-01-01;iOS;App;neg;photo;collision;3-10...,
49,54;2024-09-19;iOS;App;neg;parcours;bris_de_gla...,
51,56;2024-02-18;Android;App;neg;complexité;bris_...,
54,59;2024-06-15;iOS;Web;neg;photo;collision;3-10...,


In [20]:

t = pd.read_csv("data/commentaires_topics.csv")
print("Nb colonnes:", len(t.columns))
print("Colonnes:", t.columns.tolist())
t.head(3)


Nb colonnes: 11
Colonnes: ['texte', 'text_norm', 'sentiment_pred', 'score_sentiment', 'incident', 'categorie_principale', 'categorie_secondaire', 'theme', 'score_categorie', 'methode_categorie', 'contradiction']


,texte,text_norm,sentiment_pred,score_sentiment,incident,categorie_principale,categorie_secondaire,theme,score_categorie,methode_categorie,contradiction
0,ios app neg complexité collision an tiers moye...,ios app neg complexité collision an tiers moye...,Négatif,1.0,True,Inconnu,NaN,Inconnu,0.267939,low_confidence,False
1,ios email pos authentification collision ans j...,ios email pos authentification collision ans j...,Positif,1.0,False,Autre,NaN,Autre,0.000000,non_incident,False
2,ios web neu photo collision ans tiers faible b...,ios web neu photo collision ans tiers faible b...,Neutre,0.0,False,Autre,NaN,Autre,0.000000,non_incident,False
